# 🍏 Apple Generative Imagery Systems - LookDev LoRA Training Pipeline
### Target: SDXL 1.0 Base | Dataset: 21 Curated Tactile Cosmetic & Studio Assets
---
이 주피터 노트북은 Google Colab(GPU: T4 / V100 / A100) 환경에서 **Apple 스탠다드 미학(스튜디오 조명, 점도, SSS, 반사광)**을 가진 LoRA를 파인튜닝하는 자동화 파이프라인입니다.

**핵심 엔지니어링 Intentionality:**
- **Dual Vision Transformers (CLIP ViT-L + OpenCLIP ViT-bigG)**: 캡션의 물리/광학 텍스트를 고차원 임베딩으로 인코딩
- **Spatial Cross-Attention Transformers**: U-Net/DiT 백본의 어텐션 레이어에 저순위 적응(Rank 16, Alpha 16) 적용
- **Min-SNR Gamma = 5**: 중간 주파수 노이즈 손실에 가중치를 두어 텍스처 수렴 품질 극대화

In [ ]:
# 1. GPU 하드웨어 가속기 확인
!nvidia-smi

In [ ]:
# 2. Google Drive 마운트 (드라이브 연결 팝업이 뜨면 '허용' 클릭)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 고속 학습 라이브러리 및 의존성 패키지 설치
!git clone https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts
%cd /content/sd-scripts
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q --upgrade -r requirements.txt
!pip install -q xformers bitsandbytes accelerate

In [ ]:
# 4. Google Drive 데이터셋 자동 탐색 및 로드 (폴더 또는 .zip 자동 압축해제)
import os, shutil, zipfile

LOCAL_TRAIN_DIR = '/content/dataset/10_apl_minimal_craft'
os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)

# 드라이브 내 가능한 경로들을 순차 탐색
candidate_paths = [
    '/content/drive/MyDrive/Generative-Imagery-Systems/apple_lora_project/cropped_1024',
    '/content/drive/MyDrive/apple_lora_project/cropped_1024',
    '/content/drive/MyDrive/cropped_1024',
    '/content/drive/MyDrive/Generative-Imagery-Systems/apple_lora_project/cropped_1024_dataset.zip',
    '/content/drive/MyDrive/apple_lora_project/cropped_1024_dataset.zip',
    '/content/drive/MyDrive/cropped_1024_dataset.zip'
]

found_source = None
for path in candidate_paths:
    if os.path.exists(path):
        found_source = path
        break

if found_source:
    print(f'데이터셋 소스 발견: {found_source}')
    if found_source.endswith('.zip'):
        with zipfile.ZipFile(found_source, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_TRAIN_DIR)
    else:
        for f in os.listdir(found_source):
            if f.endswith('.png') or f.endswith('.txt'):
                shutil.copy2(os.path.join(found_source, f), os.path.join(LOCAL_TRAIN_DIR, f))
    
    files = [f for f in os.listdir(LOCAL_TRAIN_DIR) if f.endswith('.png') or f.endswith('.txt')]
    print(f'✅ 데이터셋 준비 완료: 총 {len(files)}개 파일 (이미지 21장 + 캡션 21개)')
else:
    print('⚠️ 드라이브 경로를 찾지 못했습니다. 왼쪽 사이드바 파일 탭에서 직접 dataset 압축파일을 업로드하셔도 됩니다.')

In [ ]:
# 5. SDXL LoRA 학습 실행 (2000 Steps, 20-30분 소요)
!accelerate launch --num_cpu_threads_per_process=2 sdxl_train_network.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --train_data_dir="/content/dataset" \
  --output_dir="/content/output_lora" \
  --output_name="apple_minimal_craft_sdxl_v1" \
  --resolution="1024,1024" \
  --train_batch_size=1 \
  --max_train_steps=2000 \
  --save_every_n_epochs=2 \
  --mixed_precision="fp16" \
  --save_precision="fp16" \
  --seed=42 \
  --learning_rate=1e-4 \
  --text_encoder_lr=5e-5 \
  --lr_scheduler="cosine_with_restarts" \
  --lr_warmup_steps=100 \
  --network_module=networks.lora \
  --network_dim=16 \
  --network_alpha=16 \
  --min_snr_gamma=5 \
  --gradient_checkpointing \
  --xformers \
  --sample_prompts="a photo in apl_minimal_craft style of organic cosmetic cream swirl on neutral background --w 1024 --h 1024 --d 42 --l 7.5 --s 30" \
  --sample_every_n_epochs=2

In [ ]:
# 6. 학습된 최종 LoRA 가중치(.safetensors)를 Google Drive에 백업
DEST_DIR = '/content/drive/MyDrive/Generative-Imagery-Systems/apple_lora_project/weights'
os.makedirs(DEST_DIR, exist_ok=True)

!cp /content/output_lora/*.safetensors "/content/drive/MyDrive/Generative-Imagery-Systems/apple_lora_project/weights/"
print('🎉 모든 LoRA 가중치 체크포인트가 Google Drive에 성공적으로 백업되었습니다!')